In [ ]:
reset

In [ ]:
import xarray as xr
import netCDF4 as nc
import pandas as pd
import numpy as np
import metpy.calc as mp
from metpy.units import units

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean
import cmocean.cm as cmo

# makes Jupyter output higher resolution
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

### Read in CSV files and convert to netcdfs

In [ ]:
## NH22P data
core='NH22P'
dpath0='/Users/dervlamk/Google Drive/My Drive/Research'
dpath=f'{dpath0}/GoC_Cores/Data/{core}'

## age and dDwax data
df = pd.read_csv(f'{dpath}/nh22p_dD.csv')
nh22p = df.to_xarray() # convert from pandas df to xarray ds
# Create Age Dimension
nh22p=nh22p.rename({'index': 'age'}) # rename default index
nh22p['age']=nh22p.bacon_age # replace index with median vals from bacon age model
nh22p=nh22p.drop_vars('bacon_age') # get rid of bacon_age variable
# Update Attributes: Age
nh22p.age.attrs['units'] = 'ka'
nh22p.age.attrs['mat_dated'] = '14C measurements of bulk organic carbon & tie-points between a benthic foraminiferal Uvigerina spp. d180 profile and LR04 benthic stack'
nh22p.age.attrs['source'] = 'Median Age from BACON ensemble'
# Update Attributes: dD_raw
nh22p.dD_raw.attrs['units'] = 'per mille'
nh22p.dD_raw.attrs['name'] = 'dD_raw'
nh22p.dD_raw.attrs['long_name'] = 'Raw Hydrogen Isotopic Coposition of C30 FAME'
# Update Attributes: dD_ivc
nh22p.dD_ivc.attrs['units'] = 'per mille'
nh22p.dD_ivc.attrs['name'] = 'dD_ivc'
nh22p.dD_ivc.attrs['long_name'] = 'Ice-Volume Corrected Hydrogen Isotopic Coposition of C30 FAME'
nh22p.dD_ivc.attrs['comment'] = 'Corrected based on LR04 benthic stack d18O using MATLAB icevolcorr.m function'
# Update Attributes: stdev
nh22p.dD_raw_stdev.attrs['units'] = 'per mille'
nh22p.dD_raw_stdev.attrs['name'] = 'stdev'
nh22p.dD_raw_stdev.attrs['long_name'] = 'Standard Deviation of uncorrected IRMS measurements of sedimentary C30 FAME Hydrogen Isotopic Coposition'
# Update Global Attributes
nh22p.attrs['info'] = 'Core NH22P hydrogen isotope record'


## dDprecip data
df_ = pd.read_csv(f'{dpath}/nh22p_dDp.csv', header=None)
df = df_.to_numpy() # convert from pandas df to numpy array
# Create Data Array and add to Dataset
diterations=np.arange(0,df.shape[1],1) # create index of iterations
nh22p['dD_p']=xr.DataArray(df,
                           coords={'age': nh22p.age, 'diterations': diterations},
                           dims=['age', 'diterations'],
                           attrs={'long_name': 'Hydrogen Isotopic Composition of Precipitation Calculated from dD_ivc',
                                  'units': 'per mille'},
                           name='dD_precip')


## %jas precip data
df_ = pd.read_csv(f'{dpath}/nh22p_jas.csv', header=None)
df = df_.to_numpy() # convert from pandas df to numpy array
# Create Data Array and add to Dataset
piterations=np.arange(0,df.shape[1],1) # create index of iterations
nh22p['pJAS']=xr.DataArray(df,
                           coords={'age': nh22p.age, 'piterations': piterations},
                           dims=['age', 'piterations'],
                           attrs={'long_name': 'Percent of July-August-September Rainfall',
                                  'units': 'mm day-1'},
                           name='pJAS')

## Write Netcdf
ofile = f'{dpath}/dD.nh22p.nc'
nh22p.to_netcdf(ofile, mode='w')

In [ ]:
# DSDP-480/479 data

core='DSDP480-479'
dpath0='/Users/dervlamk/Google Drive/My Drive/Research'
dpath=f'{dpath0}/GoC_Cores/Data/{core}'

## age and dDwax data
df = pd.read_csv(f'{dpath}/guaymas.csv')
dsdp = df.to_xarray() # convert from pandas df to xarray ds
# Create Age Dimension
dsdp=dsdp.rename({'index': 'age'}) # rename default index
dsdp['age']=dsdp.bacon_age # replace index with median vals from bacon age model
dsdp=dsdp.drop_vars('bacon_age') # get rid of bacon_age variable
# Update Attributes: Age
dsdp.age.attrs['units'] = 'ka'
dsdp.age.attrs['mat_dated'] = '14C measurements of bulk organic carbon & tie-points between a benthic foraminiferal Uvigerina spp. d180 profile and pollen record and LR04 benthic stack'
dsdp.age.attrs['source'] = 'Median Age from BACON ensemble'
# Update Attributes: dD_raw
dsdp.dD_raw.attrs['units'] = 'per mille'
dsdp.dD_raw.attrs['name'] = 'dD_raw'
dsdp.dD_raw.attrs['long_name'] = 'Raw Hydrogen Isotopic Coposition of C30 FAME'
# Update Attributes: dD_ivc
dsdp.dD_ivc.attrs['units'] = 'per mille'
dsdp.dD_ivc.attrs['name'] = 'dD_ivc'
dsdp.dD_ivc.attrs['long_name'] = 'Ice-Volume Corrected Hydrogen Isotopic Coposition of C30 FAME'
dsdp.dD_ivc.attrs['comment'] = 'Corrected based on LR04 benthic stack d18O using MATLAB icevolcorr.m function'
# Update Attributes: stdev
dsdp.dD_raw_stdev.attrs['units'] = 'per mille'
dsdp.dD_raw_stdev.attrs['name'] = 'stdev'
dsdp.dD_raw_stdev.attrs['long_name'] = 'Standard Deviation of uncorrected IRMS measurements of sedimentary C30 FAME Hydrogen Isotopic Coposition'
# Update Global Attributes
dsdp.attrs['info'] = 'Core DSDP-480/479 hydrogen isotope record'


## dDprecip data
df_ = pd.read_csv(f'{dpath}/guaymas_dDp.csv', header=None)
df = df_.to_numpy() # convert from pandas df to numpy array
# Create Data Array and add to Dataset
diterations=np.arange(0,df.shape[1],1) # create index of iterations
dsdp['dD_p']=xr.DataArray(df,
                          coords={'age': dsdp.age, 'diterations': diterations},
                          dims=['age', 'diterations'],
                          attrs={'long_name': 'Hydrogen Isotopic Composition of Precipitation Calculated from dD_ivc',
                                  'units': 'per mille'},
                          name='dD_precip')


## %jas precip data
df_ = pd.read_csv(f'{dpath}/guaymas_jas.csv', header=None)
df = df_.to_numpy() # convert from pandas df to numpy array
# Create Data Array and add to Dataset
piterations=np.arange(0,df.shape[1],1) # create index of iterations
dsdp['pJAS']=xr.DataArray(df,
                           coords={'age': dsdp.age, 'piterations': piterations},
                           dims=['age', 'piterations'],
                           attrs={'long_name': 'Percent of July-August-September Rainfall',
                                  'units': 'mm day-1'},
                           name='pJAS')

## Write Netcdf
ofile = f'{dpath}/dD.dsdp480-479.nc'
dsdp.to_netcdf(ofile, mode='w')

### Make Box Plots

In [ ]:
# Set Holocene and LIG age bounds [ka]
hol = 11.7
lig = [117, 130]

# extract dD_precip data from holocene & LIG
holdsdp=dsdp.dD_p[np.where(dsdp.age<=hol)].median(dim='diterations')
ligdsdp=dsdp.dD_p[np.where((dsdp.age>=lig[0]) & (dsdp.age<=lig[1]))].median(dim='diterations')

hol22p=nh22p.dD_p[np.where(nh22p.age<=hol)].median(dim='diterations')
lig22p=nh22p.dD_p[np.where((nh22p.age>=lig[0]) & (nh22p.age<=lig[1]))].median(dim='diterations')

data = [holdsdp, ligdsdp, hol22p, lig22p]

In [ ]:
flierprops = dict(marker='+', markerfacecolor='grey', markersize=5,
                  linestyle='none', markeredgecolor='grey')

font={'color':  'k',
      'weight': 'bold',
      'size': 12,
      'horizontalalignment': 'center'}
fig,ax=plt.subplots(nrows=1, ncols=1, figsize=(7,5))

# box and whisker plot that constrains whisker length to 5th & 95th percentiles
bp=ax.boxplot(data, flierprops=flierprops, whis=[5, 95])

for median in bp['medians']:
    median.set(c='indianred', lw=1.5, ls='--', zorder=0)
#for flier in bp['fliers']:
#    flier.set(marker='+', color ='k')
    
ax.text(1.5,-42,'DSDP-480/479', fontdict=font)
ax.text(3.5,-42,'NH22P', fontdict=font)
ax.set_xticklabels(['HOLOCENE', 'LIG', 'HOLOCENE', 'LIG']) # x tick labels
plt.axvline(x=2.5, linewidth=1, color='k') # add vertical line separating DSDP & NH22P data
ax.set_ylabel(u'$\delta$D$_{precip}$ [‰]') # y axis label

plt.savefig("dDprecip_d480_nh22p_LIG_vs_Holocene.pdf")

In [ ]:
diffdsdp=ligdsdp.mean()-holdsdp.mean()
diff22p=lig22p.mean()-hol22p.mean()

In [ ]:
nh22p.pJAS.median(dim='piterations').plot()